# GSB 5544 — NumPy Essentials: The 7 Concepts Behind Every DataFrame  
**SOLUTION VERSION**

Pandas is built on **NumPy**. Every column of a DataFrame is a NumPy array underneath, so understanding 7 NumPy ideas makes pandas make sense. As before, each concept answers a **question**.

**Data set:** the same `coffee_purchases.csv`. We'll pull columns *out* of pandas into NumPy to see what's really there.

In [ ]:
import numpy as np
import pandas as pd

coffee = pd.read_csv("")   # <-- put the CSV URL / path here
coffee["spend"] = coffee["amount"].abs()
coffee.head(3)

---
## Concept 1 — The `ndarray`: *What is a pandas column made of?*

A NumPy array is a grid of values that are **all the same type**. That single rule is what makes it fast. A Series is an array plus an index.

In [ ]:
spend = coffee["spend"].to_numpy()      # pull the raw array out of the Series
print(type(spend))
print(spend[:10])

In [ ]:
# Q: What do I need to know about an array?  shape, ndim, dtype, size
print("shape:", spend.shape)
print("ndim :", spend.ndim)
print("dtype:", spend.dtype)
print("size :", spend.size)

In [ ]:
# Building arrays from scratch
print(np.array([1, 2, 3]))
print(np.zeros(4))
print(np.arange(0, 10, 2))        # start, stop (exclusive), step
print(np.linspace(0, 1, 5))       # 5 evenly spaced numbers from 0 to 1

---
## Concept 2 — dtype: *Why does one text value change a whole column?*

An array has **one** dtype. Mixing types forces NumPy to pick the most general one (usually a string), which is why a single `"N/A"` in a CSV turns a numeric column into `object`.

In [ ]:
print(np.array([1, 2, 3]).dtype)
print(np.array([1, 2.5, 3]).dtype)       # int gets promoted to float
print(np.array([1, 2, "three"]).dtype)   # everything becomes a string!

In [ ]:
# Q: How do I convert?  .astype()  (same idea as pandas)
years = coffee["year"].to_numpy()
print(years.dtype)
print(years.astype(str)[:5])
print(years.astype(float)[:5])

---
## Concept 3 — Vectorization: *How do I do math on 821 numbers without a loop?*

Operations apply to **every element at once**. This is the reason `coffee["amount"] * 2` works in pandas. No `for` loop needed — and it's ~100× faster.

In [ ]:
# Q: What would each purchase cost with 8% tax?
with_tax = spend * 1.08
with_tax[:5]

In [ ]:
# Array + array works element by element (shapes must match)
amount = coffee["amount"].to_numpy()
print((amount + spend)[:5])              # negative + positive = 0 for every row

In [ ]:
# Math functions are vectorized too
print(np.round(spend, 0)[:5])
print(np.log(spend)[:5])
print(np.sqrt(spend)[:5])

In [ ]:
# Q: How much faster than a loop?
big = np.random.rand(1_000_000)

%timeit [x * 1.08 for x in big]
%timeit big * 1.08

---
## Concept 4 — Aggregation: *How do I summarize a column in one number?*

`.mean()`, `.sum()`, `.min()`, `.max()`, `.std()` on a Series are NumPy aggregations. Also learn `argmax`/`argmin` — *where* the max is — which is the idea behind pandas' `idxmax`.

In [ ]:
# Q: What's the total, average, and biggest coffee purchase?
print("total :", spend.sum().round(2))
print("mean  :", spend.mean().round(2))
print("median:", np.median(spend))
print("max   :", spend.max())
print("std   :", spend.std().round(2))

In [ ]:
# Q: WHICH purchase was the biggest?  argmax gives the position
i = spend.argmax()
print(i)
coffee.iloc[i]

In [ ]:
# Q: What are the quartiles?  (pd.qcut used these under the hood)
np.percentile(spend, [25, 50, 75])

---
## Concept 5 — Boolean masks: *How do I pick out the rows that meet a condition?*

A comparison on an array returns an array of `True`/`False` — a **mask**. Indexing with a mask keeps the `True` rows. This is exactly what `coffee[coffee["amount"] < -10]` does in pandas.

In [ ]:
# Q: Which purchases were over $10?
mask = spend > 10
print(mask[:10])
print("how many True:", mask.sum())        # True counts as 1
print("fraction     :", mask.mean().round(3))

In [ ]:
spend[mask][:10]           # keep only the True positions

In [ ]:
# Combine conditions: & (and), | (or), ~ (not). Parentheses required.
dow = coffee["day_of_week"].to_numpy()
weekend_big = (spend > 10) & ((dow == "Saturday") | (dow == "Sunday"))
coffee[weekend_big]

In [ ]:
# Q: Where (which positions) are the True values?
np.where(spend > 15)[0]

In [ ]:
# np.where as an if/else on every element  -> a new categorical variable
size = np.where(spend > 10, "large", "small")
pd.Series(size).value_counts()

---
## Concept 6 — Indexing & slicing 2-D arrays: *Where do `.iloc[rows, cols]` rules come from?*

A DataFrame's numbers are a 2-D array. `arr[rows, cols]` with integers and `:` slices is exactly `.iloc`. Position-based, end-exclusive.

In [ ]:
nums = coffee[["amount", "year", "spend"]].to_numpy()
print(nums.shape)
nums[:3]

In [ ]:
print(nums[0, 2])          # row 0, column 2  (like .iloc[0, 2])
print(nums[0:3, 0])        # rows 0-2, column 0
print(nums[:, 1][:5])      # all rows, column 1
print(nums[-2:, :])        # last two rows, all columns

In [ ]:
# Reshaping: the same 12 numbers as 3x4 or 4x3
a = np.arange(12)
print(a.reshape(3, 4))
print(a.reshape(4, 3).T)   # .T = transpose

---
## Concept 7 — Missing values (`NaN`): *Why does my average come out as `nan`?*

`np.nan` is a float that means "missing". Plain NumPy math **propagates** it; pandas methods **skip** it. Knowing this explains why `Series.mean()` and `np.mean()` can disagree.

In [ ]:
x = np.array([4.5, np.nan, 3.0, 7.25])

print(np.mean(x))          # nan  -> one missing value poisons the result
print(np.nanmean(x))       # 4.9166 -> ignore the missing value
print(pd.Series(x).mean()) # pandas skips NaN by default

In [ ]:
# Q: How do I find missing values?  (nan != nan, so use isnan)
print(x == np.nan)         # all False -- the trap
print(np.isnan(x))         # the right way
print(np.isnan(x).sum(), "missing")

In [ ]:
# Q: Does the coffee data have any missing values?
coffee.isna().sum()

In [ ]:
# Filling or dropping
print(np.nan_to_num(x, nan=0))
print(x[~np.isnan(x)])

---
## Summary — 7 NumPy concepts and the pandas feature they explain

| # | NumPy concept | Question | pandas feature it explains |
|---|---|---|---|
| 1 | `ndarray` (shape, ndim, dtype) | What is a column made of? | Series = array + index |
| 2 | one dtype per array | Why does one text value change a column? | `object` columns, `.astype()` |
| 3 | vectorization | Math on 821 values without a loop? | `df["a"] * 2`, `df["a"] + df["b"]` |
| 4 | aggregation, `argmax` | Summarize in one number? Where is the max? | `.mean()`, `.sum()`, `.idxmax()` |
| 5 | boolean masks, `np.where` | Pick rows that meet a condition? | `df[df["a"] > 10]`, `&`, `\|`, `~` |
| 6 | 2-D indexing `[rows, cols]` | Where do `.iloc` rules come from? | `.iloc[r, c]`, end-exclusive slices |
| 7 | `NaN` | Why is my mean `nan`? | `.isna()`, `skipna=True`, `.fillna()` |